# Reference QC, foreign tracts & ghost detection

Three diagnostics that fall out of the same machinery as painting. All three read a
**leave-one-out** view of the genealogy — what the rest of the tree says about a
haplotype *ignoring its own label* (`tspaint.output.loo_posterior_table`) — because the
ordinary down-pass posterior hides a sample's foreign tracts behind its own (confident)
label.

- **Reference QC** — audit a labelled panel for admixed / mislabelled references.
- **Anonymous foreign tracts** — flag foreign tracts of any haplotype, without naming a donor.
- **Ghost-source detection** — find tracts from a population *not in the panel* (deep outliers).


## 1. Reference QC

Simulate a panel where a few nominal `A` / `B` references are themselves slightly admixed,
then audit it. `reference_qc` ranks references by credibility (least-credible first) and
maps each one's own introgression. It assumes the clean references are the **majority**.


In [ ]:
import numpy as np
import tspaint

ts = tspaint.sim.simulate_admixture_impure_refs(
    n_admix=2, n_pure=8, n_impure=3, sequence_length=2e6,
    ref_impurity=0.3, Ne=1000, T_admix=150, T_split=5000, random_seed=1)

node_pop = ts.tables.nodes.population
names = {p: ts.population(p).metadata.get('name', str(p)) for p in range(ts.num_populations)}
pid = {n: p for p, n in names.items()}
of = lambda nm: [int(s) for s in ts.samples() if node_pop[s] == pid[nm]]
labels = {s: 0 for s in of('A') + of('RA')}        # RA = impure-A panel (nominally A)
labels.update({s: 1 for s in of('B') + of('RB')})


In [ ]:
qc = tspaint.reference_qc(ts, labels)
# least-credible references first -- the impure ones float to the top of the audit
for row in qc.summary()[:6]:
    print(f"ref {row['ref']:>3}  label {row['label']}  credibility {row['credibility']:.2f}"
          f"  anchor {row['is_anchor']!s:>5}  foreign_frac {row['foreign_fraction']:.2f}")


In [ ]:
# the least-credible reference's own foreign tracts (its introgression map, thresholded)
suspect = qc.summary()[0]['ref']
print('suspect reference:', suspect)
print('flagged foreign tracts:', qc.flagged_tracts(suspect, deadband=0.3))


## 2. Anonymous foreign tracts

`foreign_tracts` flags foreign tracts of any haplotype **without attributing a donor**.
For a labelled reference it reports where the genealogy withholds support from its label;
for an unlabelled query it reports where the tract fits *no* panel source.


In [ ]:
impure = of('RA') + of('RB')
ft = tspaint.foreign_tracts(ts, labels, impure, soft_refs=set(impure), min_score=0.4)
flagged = {r: ft[r] for r in impure if ft[r]}      # references carrying a foreign tract
print(f'{len(flagged)}/{len(impure)} impure references carry a flagged foreign tract')
for r, tracts in list(flagged.items())[:3]:
    print(f'  sample {r}:', [(round(l), round(rr), round(sc, 2)) for (l, rr, sc) in tracts])


## 3. Ghost-source detection

An *unsampled* third source (`GHOST`) contributes to the admixed queries as a deep outgroup.
`detect_ghost` flags tracts that fit no panel source **and** sit on an anomalously deep
branch — the genome-wide **burden** separates ghost from the matched no-ghost control ~10x.
This is detection (*that* a tract is foreign-to-the-panel), not attribution.


In [ ]:
def ghost_burden(ghost_fraction, seed=1):
    ts = tspaint.sim.simulate_admixture_with_ghost(
        n_admix=12, n_ref=8, sequence_length=2e6, ghost_fraction=ghost_fraction,
        T_admix=100, T_split_AB=2000, T_split_ABC=20000, Ne=1000, random_seed=seed)
    npop = ts.tables.nodes.population
    nm = {p: ts.population(p).metadata.get('name', str(p)) for p in range(ts.num_populations)}
    pg = {n: p for p, n in nm.items()}
    of = lambda x: [int(s) for s in ts.samples() if npop[s] == pg[x]]
    q = of('ADMIX'); lab = {s: 0 for s in of('A')}; lab.update({s: 1 for s in of('B')})
    gh = tspaint.detect_ghost(ts, lab, q)
    return float(np.mean([gh['burden'][x] for x in q]))

print('ghost burden (25% ghost):', round(ghost_burden(0.25), 3))
print('ghost burden (no ghost) :', round(ghost_burden(0.0), 3))


---

**When these pay off.** The signal is *deep* divergence of the foreign source relative to
the admixture age — strongest for archaic-like introgression and clean panels with many
anchors; it vanishes when the donor is a close cousin or the admixture is ancient relative
to a shallow source split (CLAUDE.md §6, §9).
